# 05 · Regression — California housing

**Dataset:** `data/housing_raw.csv` — California housing (20,640 rows) with a synthetic
`Neighbourhood`, `HouseType` and `Condition`, plus some injected mess.
**Target:** `MedianHouseValue` (median house value per block, in $100k).
**Covers:** the whole pipeline again, but for **regression** — where the metrics,
the target distribution and the diagnostics all differ.
**Time yourself:** ~45 minutes.

Classification questions ("what's the survival rate?") get replaced by
"what's the average price per neighbourhood?" — this is the dataset shape a
house-price interview actually uses.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

df = pd.read_csv('data/housing_raw.csv')
print(df.shape)
df.head()

**Column reference**

| column | meaning |
|---|---|
| `MedInc` | median household income in the block, in **dollars** |
| `HouseAge` | median house age in the block |
| `AveRooms` / `AveBedrms` | average rooms / bedrooms per household |
| `Population` | block population |
| `AveOccup` | average household occupancy |
| `Latitude` / `Longitude` | block centroid |
| `Neighbourhood`, `HouseType`, `Condition` | synthetic categoricals |
| `MedianHouseValue` | **target** — median house value, in $100k |

---

## Part A — Clean

### Q1. First look: shape, dtypes, missing values, duplicates. List every problem you find before fixing any of them.

<details><summary>hint</summary>

`.info()`, `.describe()`, `.duplicated().sum()`, `.isnull().mean()`. Look at `Condition.unique()` too.

</details>

In [ ]:
# your code here

### Q2. Fix them all: drop duplicates, parse `MedInc` to numeric (mind the `$` and the thousands separators), normalise `Condition`.

<details><summary>hint</summary>

Same moves as notebook 01: strip separators, then `pd.to_numeric(..., errors='coerce')`; `.str.strip().str.lower()` for the categorical.

</details>

In [ ]:
# your code here

### Q3. Look hard at `MedianHouseValue`'s maximum and count how many rows sit exactly at it.
What has been done to this data, and what does it mean for your model?

<details><summary>hint</summary>

Sort the target descending and look at how many rows share the top value. It's not a coincidence.

</details>

In [ ]:
# your code here

### Q4. Handle the missing `MedInc`, `HouseAge` and `AveRooms`. Choose mean or median per
column and justify with the skew. (Just do it globally for now — you'll redo it
leak-free in the pipeline later.)

<details><summary>hint</summary>

Check `.skew()` first. Rule of thumb: |skew| > 1 → median.

</details>

In [ ]:
# your code here

---

## Part B — EDA & the questions they actually ask

### Q5. **The classic warm-up:** what is the average house value per neighbourhood?
Sort descending, and include the count per group.

<details><summary>hint</summary>

`groupby('Neighbourhood')['MedianHouseValue'].agg(...)`. Always report the count next to the mean.

</details>

In [ ]:
# your code here

### Q6. Follow-up they always ask: the **top 3 neighbourhoods by average value, but only
those with at least 500 blocks**, and show the average income alongside.

<details><summary>hint</summary>

Named aggregation `agg(n=('col','size'), avg=('col','mean'))`, then `.query()` to filter groups, then `.nlargest(3, 'avg_value')`.

</details>

In [ ]:
# your code here

### Q7. Cross two categoricals: average value by `Neighbourhood` × `Condition`, as a pivot
table. Does `Condition` matter? (Careful — you know how this column was made.)

<details><summary>hint</summary>

`pivot_table(index=..., columns=..., values=..., aggfunc='mean')`. Then ask whether the column-to-column variation is bigger than noise.

</details>

In [ ]:
# your code here

### Q8. Which numeric feature correlates most with the target? Show a heatmap, then a
scatterplot of the strongest pair.

<details><summary>hint</summary>

`corr['MedianHouseValue'].sort_values()`. Sample before scattering 20k points — it's faster and less of an ink blob.

</details>

In [ ]:
# your code here

### Q9. `AveRooms` has skew ~20. Look at its extreme values — error or real? Then decide
what to do and act.

<details><summary>hint</summary>

Look at `Population` and `AveOccup` on those rows before calling it an error. The clue is in the denominator.

</details>

In [ ]:
# your code here

---

## Part C — Features

### Q10. Engineer three features that should beat the raw columns:
`rooms_per_person`, `bedrooms_ratio` (bedrooms as a share of rooms), and
`households` (how many households are in the block).
Check their correlation with the target.

One of these three turns out to be nearly useless. Which, and why?

<details><summary>hint</summary>

Ratios of existing columns: `bedrooms_ratio = AveBedrms / AveRooms`, `households = Population / AveOccup`. Then compare the two ratios against the raw count and ask what each actually measures.

</details>

In [ ]:
# your code here

### Q11. The target is right-skewed. Compare `MedianHouseValue` against `log1p` of it: plot
both and print both skews. Would you train on the log target? What breaks if you do?

<details><summary>hint</summary>

`np.log1p` / `np.expm1`. The trap is in what happens to your reported metric.

</details>

In [ ]:
# your code here

---

## Part D — Model

### Q12. Split 80/20 with `random_state=42`. Then answer: why is there no `stratify=` here,
and is a random split even appropriate for this data?

<details><summary>hint</summary>

`stratify` needs discrete classes. Then think about what Latitude/Longitude imply about neighbouring rows.

</details>

In [ ]:
# your code here

### Q13. Build a `ColumnTransformer` + `Pipeline` and fit a `LinearRegression` baseline.
Report RMSE, MAE and R² on test.

<details><summary>hint</summary>

Same shape as notebook 03. RMSE = `mean_squared_error(...) ** 0.5`. Translate it into dollars — the target is in $100k.

</details>

In [ ]:
# your code here

### Q14. Explain the difference between RMSE and MAE **using your own two numbers above**.
Which would you report to a business stakeholder pricing houses?

<details><summary>hint</summary>

RMSE squares the errors before averaging, so it is dominated by the worst ones. Which one is a sentence a human can act on?

</details>

In [ ]:
# your code here

### Q15. Fit a `RandomForestRegressor` and a `HistGradientBoostingRegressor`. Compare all three
models. How much did the non-linear models buy you over linear?

<details><summary>hint</summary>

`HistGradientBoostingRegressor` is sklearn's built-in LightGBM-alike — fast and usually the strongest thing you have without installing xgboost.

</details>

In [ ]:
# your code here

### Q16. Cross-validate the winner with 5-fold `KFold` (`scoring='neg_root_mean_squared_error'`).
Why `neg_`?

<details><summary>hint</summary>

sklearn scorers are always 'higher is better'. RMSE isn't, so it's negated.

</details>

In [ ]:
# your code here

---

## Part E — Diagnose

### Q17. Plot predicted vs actual, and residuals vs predicted, for the boosting model.
Name **two** specific pathologies visible in these plots.

<details><summary>hint</summary>

Residuals should look like a shapeless cloud around zero. Anything with structure — a diagonal edge, a widening fan — is the model telling you something it can't handle.

</details>

In [ ]:
# your code here

### Q18. Check train vs test R² for the random forest. Over- or underfitting? Then fix it and
show the fix worked.

<details><summary>hint</summary>

Compare `r2_score(y_train, model.predict(X_train))` against test. A gap over ~0.1 with the train score near 1.0 is the tell.

</details>

In [ ]:
# your code here

### Q19. Get permutation importance on the test set. Does `Condition` show up as unimportant,
as you predicted in Part B? What does `Latitude`/`Longitude` ranking high tell you?

<details><summary>hint</summary>

`permutation_importance(model, X_test, y_test, scoring='r2')`. Look at where `Condition` lands, and remember how it was generated.

</details>

In [ ]:
# your code here

---

## Debrief — what makes this a *regression* interview

- **Look at the target first.** The 5.0 cap is the whole test here. Candidates who plot the
  target find it in 30 seconds; candidates who go straight to `.fit()` never do.
- **Metrics differ.** No accuracy, no AUC. RMSE vs MAE is a real choice with a real
  argument, and R² is the one number a stakeholder understands.
- **Residual plots are the regression equivalent of a confusion matrix.** Structure in the
  residuals = something the model can't represent.
- **Non-linearity actually pays here** (R² 0.66 → 0.83), unlike Titanic. Knowing *when*
  the fancy model earns its keep is the skill — not always reaching for it.
- **The split is a judgment call.** Spatial autocorrelation means the honest number is
  lower than the one you'll report. Say so before they ask.